# 0.5.2 - XGBoost Spatial Evaluation

Does the model generalise to municipalities without any measurement? Cross-validation over all observations with two groupings:

- **Held-out buildings**: same grouping as the train/test split (neighbouring buildings can be in training).
- **Held-out municipalities**: every measurement of a municipality is in the same fold.

Both use the hyperparameters tuned in 0.5 (no retuning per fold) and the thresholds saved by 0.5.1. The hyperparameters were tuned on the training set, so the scores are slightly optimistic for both groupings; the difference between them is what matters.

## Setup

In [ ]:
import json
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.base import clone
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold

from evaluation import load_test_egid, load_test_set, operating_points, regression_scores

ROOT = '../../'
DIR = '../../models/'
RANDOM_SEED = 42
N_FOLDS = 5

## Data and settings

In [ ]:
X_test, y_test = load_test_set(ROOT)
X_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-x-train.csv', index_col=0).replace([np.inf, -np.inf], np.nan)
y_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-y-train.csv', index_col=0).squeeze()
egid_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-egid-train.csv', index_col=0)['EGID']

X = pd.concat([X_train, X_test])
y = pd.concat([y_train, y_test])
egid = pd.concat([egid_train, load_test_egid(ROOT)]).loc[X.index]
municipality = X['Rdn - Commune']

# Unfitted copy of the 0.2 pipeline, refitted in each fold
with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    preprocessor = clone(pickle.load(file))
with open(f'{DIR}0.5-xgboost-best-params.json') as file:
    best_params = json.load(file)
with open(f'{DIR}0.5-decision-threshold.json') as file:
    settings = json.load(file)
RADON_REFERENCE_LEVEL = settings['reference_level']
THRESHOLDS = {f'Map {cutoff}': threshold for cutoff, threshold in settings['map_matched_thresholds'].items()}

X.shape, egid.nunique(), municipality.nunique()

## Cross-validation

In [ ]:
def out_of_fold_predictions(groups):
    """Predictions (Bq/m3) for every observation, from a model trained on the other folds."""
    predictions = pd.Series(np.nan, index=y.index)
    for fit_idx, held_idx in GroupKFold(n_splits=N_FOLDS).split(X, y, groups):
        fold_preprocessor = clone(preprocessor).fit(X.iloc[fit_idx])
        model = xgb.XGBRegressor(**best_params, objective='reg:squarederror', n_jobs=-1, random_state=RANDOM_SEED)
        model.fit(fold_preprocessor.transform(X.iloc[fit_idx]), np.log1p(y.iloc[fit_idx]))
        predictions.iloc[held_idx] = np.expm1(model.predict(fold_preprocessor.transform(X.iloc[held_idx])))
    return predictions


predictions = {
    'Held-out buildings': out_of_fold_predictions(egid),
    'Held-out municipalities': out_of_fold_predictions(municipality),
}

## Results

In [ ]:
exceeds = y >= RADON_REFERENCE_LEVEL
pd.DataFrame([
    {'Grouping': name, **regression_scores(y, pred), 'Average precision': round(average_precision_score(exceeds, pred), 3)}
    for name, pred in predictions.items()
])

In [ ]:
pd.concat([operating_points(exceeds, pred, THRESHOLDS).assign(Grouping=name) for name, pred in predictions.items()], ignore_index=True)